# Pair-level preprocessing for HH->bbtautau: (jet, tau) overlap dataset from .root files

This notebook builds the labelled, preprocessed **train / validation / test** datasets for the objective 3.1 pair-classification task (FF / FT / TF / TT), starting directly from the `.root` ntuples in `overlap_resolver/root_datasets/HH_bbtt`.

It reuses:
- `flavour_tag_ml` (repo `flavour_tagging`): generic checkpoint I/O and standardization utilities (`fit_transform_standardize`, `transform_standardize`, `compute_feature_stats`, `JetDataset`, ...).
- `overlap_resolver/src`: `obj_3_1.py` (file loading + analysis-level selection), `overlap_kinematics.py` (pair kinematics), `truth_vs_reco_params.py` (truth labelling), and `overlap_pairs_dataset_builder.py` (this notebook's main orchestrator, `build_pair_dataset_from_root` + `split_pairs_by_event`).

Pipeline:
1. **STEP 1** — build the (jet, tau) pair dataset from all `.root` files, with geometric overlap slicing and truth labelling, checkpointed chunk by chunk.
2. **STEP 2** — merge the checkpoint chunks into a single (X, y, event_id) array.
3. **STEP 3** — split into train / val / test **by event** (no data leakage).
4. **STEP 4** — materialize each split to disk.
5. **STEP 5** — standardize features (fit on train, transform val/test).
6. **STEP 6** — final sanity checks (shapes, 4-class balance, NaN/Inf, mean/std).
7. *(Optional)* — wrap the splits in `JetDataset` / `DataLoader`.


## 0. Setup

In [ ]:
# MOUNT GOOGLE DRIVE
#
# Used ONLY to persist the produced datasets (X/y/event_id splits,
# normalization params, ...) across Colab sessions. The .root INPUT files
# are not read from Drive: they come from the cloned "overlap_resolver" repo.

from google.colab import drive
drive.mount("/content/drive/")


In [ ]:
# IMPORT GITHUB REPO: flavour_tag_ml (repo "flavour_tagging")
#
# Contains generic, reusable pieces of the pipeline: checkpoint_io.py,
# merge_datasets.py, data.py (JetDataset, standardization helpers), utils.py.

from google.colab import userdata
import os
import sys

token = userdata.get("GITHUB_TOKEN")

REPO_PATH_FLAVOUR_TAG_ML = "/content/flavour_tagging"

if not os.path.exists(REPO_PATH_FLAVOUR_TAG_ML):
    !git clone https://{token}@github.com/giumont/flavour_tagging.git {REPO_PATH_FLAVOUR_TAG_ML}
else:
    !git -C {REPO_PATH_FLAVOUR_TAG_ML} pull --rebase

# The importable package lives under <repo>/src/flavour_tag_ml, so we add
# <repo>/src (not the repo root) to sys.path and import it as a top-level
# package: "import flavour_tag_ml".
SRC_PATH_FLAVOUR_TAG_ML = os.path.join(REPO_PATH_FLAVOUR_TAG_ML, "src")
if SRC_PATH_FLAVOUR_TAG_ML not in sys.path:
    sys.path.append(SRC_PATH_FLAVOUR_TAG_ML)

import flavour_tag_ml


In [ ]:
# IMPORT GITHUB REPO: overlap_resolver
#
# Contains, under "src/": obj_3_1.py, overlap_kinematics.py,
# truth_vs_reco_params.py and overlap_pairs_dataset_builder.py (this
# notebook's main orchestrator). The input .root files live under
# "root_datasets/HH_bbtt/" in this same repo. This is also, under
# "notebooks/", where this notebook itself is meant to live.
#
# NOTE: the original snippet reassigned "repo_path" to the SAME path used
# for flavour_tagging ("/content/flavour_tagging") -- fixed here: this repo
# gets its own clone path, REPO_PATH_OVERLAP_RESOLVER.

token_overlap = userdata.get("GITHUB_TOKEN_OVERLAP_RESOLVER")

REPO_PATH_OVERLAP_RESOLVER = "/content/overlap_resolver"

if not os.path.exists(REPO_PATH_OVERLAP_RESOLVER):
    !git clone https://{token_overlap}@github.com/giumont/overlap_resolver.git {REPO_PATH_OVERLAP_RESOLVER}
else:
    !git -C {REPO_PATH_OVERLAP_RESOLVER} pull --rebase

# Unlike flavour_tag_ml, the modules under overlap_resolver/src are FLAT:
# they import each other directly (e.g. overlap_kinematics.py does
# "from obj_3_1 import ..." and "from truth_vs_reco_params import ...",
# NOT "from src.obj_3_1 import ..."). To keep them working unmodified, we
# append the "src" folder ITSELF to sys.path (not the repo root), and
# import them below as flat top-level modules.
SRC_PATH_OVERLAP_RESOLVER = os.path.join(REPO_PATH_OVERLAP_RESOLVER, "src")
if SRC_PATH_OVERLAP_RESOLVER not in sys.path:
    sys.path.append(SRC_PATH_OVERLAP_RESOLVER)


In [ ]:
# IMPORT PROJECT FUNCTIONS

# --- flavour_tag_ml: proper package, "from flavour_tag_ml.<module> import ..." ---
from flavour_tag_ml.data import (
    JetDataset,
    fit_transform_standardize,
    transform_standardize,
    compute_feature_stats,
)
import flavour_tag_ml.checkpoint_io as ft_checkpoint_io  # kept for reference/debug only
from flavour_tag_ml.merge_datasets import merge_checkpoint_chunks, check_checkpoint_progress
from flavour_tag_ml.utils import set_seed, load_mmap

# --- overlap_resolver/src: FLAT modules (see note in the cell above) ---
import obj_3_1
import overlap_kinematics
import truth_vs_reco_params
import overlap_pairs_dataset_builder as pair_builder


In [ ]:
# RELOAD MODULES
#
# Convenience for development only (picks up edits made to the .py files
# during this Colab session); harmless, and a no-op in effect, otherwise.
# Reload order matters: truth_vs_reco_params is reloaded BEFORE
# overlap_kinematics, since the latter imports names FROM the former at
# import time (see overlap_kinematics.py top-level imports).

import importlib

importlib.reload(flavour_tag_ml.data)
importlib.reload(flavour_tag_ml.checkpoint_io)
importlib.reload(flavour_tag_ml.merge_datasets)
importlib.reload(flavour_tag_ml.utils)

importlib.reload(obj_3_1)
importlib.reload(truth_vs_reco_params)
importlib.reload(overlap_kinematics)
importlib.reload(pair_builder)


In [ ]:
# STANDARD PYTHON LIBRARIES

import gc
import shutil

import numpy as np
import awkward as ak
import torch
from torch.utils.data import DataLoader


In [ ]:
# (OPTIONAL) Manual inspection of one .root file
#
# Lists the branches available in the TTree, purely as a sanity check
# before filling in the branch names in GLOBAL PARAMS below.
# Set CHECK_FILE = None to skip this cell entirely.

import uproot

CHECK_FILE = None
# e.g.: os.path.join(REPO_PATH_OVERLAP_RESOLVER, "root_datasets", "HH_bbtt",
#                     "output_GGF_mc23a_bypass_noOR_000001.root")
CHECK_TREE_NAME = "AnalysisMiniTree"

if CHECK_FILE is not None:
    with uproot.open(CHECK_FILE) as root_file:
        tree = root_file[CHECK_TREE_NAME]
        branch_names = sorted(tree.keys())
        print(f"Tree: {CHECK_TREE_NAME}  |  entries: {tree.num_entries}  |  branches: {len(branch_names)}")
        for name in branch_names:
            print(" ", name)


## 1. Global parameters

All configurable constants in one block, ALL_CAPS, English comments.

**Branches flagged `# TODO CONFIRM`**: I could not verify these exact branch names against the actual `.root` files (no ROOT file / uproot listing was available to me), so they're best-effort guesses based on naming conventions in the other scripts. Please run the optional "manual inspection" cell above (set `CHECK_FILE` to a real path) and confirm/correct them before running STEP 1 for real — a wrong branch name will simply raise a `KeyError` in `tree.arrays(...)`, so it's safe but wasteful to run with an unconfirmed name.

In [ ]:
# =============================================================================
# GLOBAL PARAMETERS
# =============================================================================

# --- Paths ----------------------------------------------------------------
# Input .root files: read directly from the cloned "overlap_resolver" repo
ROOT_INPUT_DIR = os.path.join(REPO_PATH_OVERLAP_RESOLVER, "root_datasets", "HH_bbtt")

# Output: datasets persisted on Google Drive so they survive across Colab sessions
DATASETS_PATH = "/content/drive/MyDrive/overlap_resolver_datasets/obj_3_1_pairs"
os.makedirs(DATASETS_PATH, exist_ok=True)

# --- obj_3_1.py module-level attributes, OVERRIDDEN here (not by editing the file) ---
# load_files() re-reads ROOT_DIR/FILE_PREFIX/FILE_SUFFIX/TREE_NAME/N_ENTRIES_CAP
# from the obj_3_1 module namespace at EVERY call (not at import time), so
# reassigning them on the imported module is enough to redirect it.
obj_3_1.ROOT_DIR = Path(ROOT_INPUT_DIR)
obj_3_1.FILE_PREFIX = "output_GGF_mc23a_bypass_noOR_0000"
obj_3_1.FILE_SUFFIX = ".root"
obj_3_1.TREE_NAME = "AnalysisMiniTree"
obj_3_1.N_ENTRIES_CAP = None  # None = all events in every file

# --- Analysis-level selection branches (jet / tau) -------------------------
JET_ANALYSIS_BRANCH = "recojet_antikt4PFlow_isAnalysisJet___NOSYS"
TAU_ANALYSIS_BRANCH = "tau_isAnalysisTau___NOSYS"

# --- Kinematic branches used to build the (jet, tau) pairs -----------------
# (order matches build_pair_dataset_from_root's signature)
JET_ETA_BRANCH = "recojet_antikt4PFlow_eta"
JET_PHI_BRANCH = "recojet_antikt4PFlow_phi"
JET_PT_BRANCH = "recojet_antikt4PFlow_pt___NOSYS"
JET_MASS_BRANCH = "recojet_antikt4PFlow_m___NOSYS"
JET_N_MUONS_BRANCH = "recojet_antikt4PFlow_nMuonSegments"  # TODO CONFIRM exact branch name

TAU_ETA_BRANCH = "tau_eta"
TAU_PHI_BRANCH = "tau_phi"
TAU_PT_BRANCH = "tau_pt___NOSYS"
TAU_NPRONG_BRANCH = "tau_nTracks"       # TODO CONFIRM exact branch name
TAU_DECAYMODE_BRANCH = "tau_DecayMode"  # TODO CONFIRM exact branch name
TAU_CHARGE_BRANCH = "tau_charge"

# --- Truth-label branches (consumed by jet_truth_label_fn / tau_truth_label_fn below) ---
# Same branches/values as truth_vs_reco_params.py (JET_TRUTH_LABEL_BRANCH,
# JET_TRUTH_LABEL_B_VALUE, TAU_TRUTH_MATCH_BRANCH), copied here as explicit
# GLOBAL PARAMS rather than imported, since build_pair_dataset_from_root
# takes the labelling logic as a callable (jet_truth_label_fn/tau_truth_label_fn),
# not as a module-level constant.
JET_TRUTH_LABEL_BRANCH = "recojet_antikt4PFlow_HadronConeExclTruthLabelID"
JET_TRUTH_LABEL_B_VALUE = 5
TAU_TRUTH_MATCH_BRANCH = "tau_truth_IsHadronicTau"

# --- Geometric overlap threshold --------------------------------------------
DR_THR = 0.4

# --- Feature columns of X, among the keys returned by build_pair_kinematics_and_labels ---
FEATURE_KEYS = [
    "pair_dr", "pair_deta", "pair_dphi", "pair_pt_ratio",
    "jet_pt", "jet_eta", "jet_phi", "jet_mass", "jet_n_muons",
    "tau_pt", "tau_eta", "tau_phi", "tau_nProng", "tau_decayMode", "tau_charge",
]

# --- Pair truth-label encoding (FF/FT/TF/TT -> int) -------------------------
LABEL_INDEX_MAP = {"FF": 0, "FT": 1, "TF": 2, "TT": 3}

# --- Train / val / test split fractions (computed on unique EVENTS, not pairs) ---
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-8, "fractions must sum to 1"

RANDOM_SEED = 42

# --- Checkpoint / manifest base name for the pair-building step -------------
PAIR_SAVE_NAME = "obj_3_1_pairs"


## 2. Set global seed

In [ ]:
# SET GLOBAL SEED (reproducibility across numpy / torch / python `random`)
set_seed(RANDOM_SEED)


## 3. STEP 1 — Build the pair dataset from `.root` files (checkpointed)

**Caveat on "resuming"** (flagging this explicitly, since it differs from the H5 pipeline's checkpointing): `build_pair_dataset_from_root` always loops over **all** files returned by `load_files()` and writes chunks starting again from `chunk_idx=0` / `event_offset=0` — it does **not** detect an existing manifest and skip already-processed files. So re-running the cell below after an interruption will **overwrite** the previous checkpoint from scratch (each file is still fast to re-read, so for 14 files this is a correctness-over-efficiency tradeoff, not a bug) rather than resume mid-way. If you want true cross-session resumption I can add that, but it's not implemented in `overlap_pairs_dataset_builder.py` as given — let me know.

In [ ]:
# =============================================================================
# STEP 1a — truth-label closures for jet / tau
#
# Parametric equivalent of label_jets_and_taus (truth_vs_reco_params.py),
# restricted to TRUTH_MODE_TAU="label" (the geometric-matching branch of the
# original function is not exposed here, since build_pair_dataset_from_root
# expects a single bool array per closure, not the (label, dr_truth) pair
# that the geometric mode returns).
#
# Required signature: fn(events_full, selection_mask) -> awkward.Array (bool)
# already indexed by the analysis-level selection mask, i.e. same jagged
# structure as jet_sel / tau_sel themselves.
# =============================================================================

def jet_truth_label_fn(a, jet_sel):
    """True jet = HadronConeExclTruthLabelID == JET_TRUTH_LABEL_B_VALUE (b-jet)."""
    flavour = a[JET_TRUTH_LABEL_BRANCH][jet_sel]
    return flavour == JET_TRUTH_LABEL_B_VALUE


def tau_truth_label_fn(a, tau_sel):
    """True tau = tau_truth_IsHadronicTau != 0."""
    return a[TAU_TRUTH_MATCH_BRANCH][tau_sel] != 0


In [ ]:
# =============================================================================
# STEP 1b — build the (jet, tau) pair dataset from all .root files
# (checkpointed chunk by chunk: one chunk per .root file)
# =============================================================================

build_result = pair_builder.build_pair_dataset_from_root(
    save_path=DATASETS_PATH,
    save_name=PAIR_SAVE_NAME,
    jet_analysis_branch=JET_ANALYSIS_BRANCH,
    tau_analysis_branch=TAU_ANALYSIS_BRANCH,
    jet_eta_branch=JET_ETA_BRANCH,
    jet_phi_branch=JET_PHI_BRANCH,
    jet_pt_branch=JET_PT_BRANCH,
    jet_mass_branch=JET_MASS_BRANCH,
    jet_n_muons_branch=JET_N_MUONS_BRANCH,
    tau_eta_branch=TAU_ETA_BRANCH,
    tau_phi_branch=TAU_PHI_BRANCH,
    tau_pt_branch=TAU_PT_BRANCH,
    tau_nProng_branch=TAU_NPRONG_BRANCH,
    tau_decayMode_branch=TAU_DECAYMODE_BRANCH,
    tau_charge_branch=TAU_CHARGE_BRANCH,
    jet_truth_label_fn=jet_truth_label_fn,
    tau_truth_label_fn=tau_truth_label_fn,
    feature_keys=FEATURE_KEYS,
    dr_thr=DR_THR,
    label_index_map=LABEL_INDEX_MAP,
    verbose=True,
)

build_result


In [ ]:
# STEP 1c — checkpoint progress check (pair-specific: no a-priori n_total,
# so this only reports how many pairs/chunks have been saved so far)
progress = pair_builder.check_pair_checkpoint_progress(DATASETS_PATH, PAIR_SAVE_NAME)
progress


## 4. STEP 2 — Merge the checkpoint chunks

`merge_pair_checkpoint_chunks` (pair-specific equivalent of `merge_checkpoint_chunks` used in the H5 pipeline) reads back every per-file chunk written in STEP 1 plus the manifest, and returns the full, already-concatenated `X`, `y`, `event_id`, `feature_names` in memory.

**Scale assumption (point 6 from the proposal):** the (jet, tau) pair dataset is assumed small enough to fit in RAM after the merge (unlike the H5 jet pipeline, which needed memmap). If this turns out to be false for the full 14-file HH_bbtt sample, this step (and STEP 4 below) should be rewritten with `np.lib.format.open_memmap`, exactly as in the H5 notebook. `output_file=""` is passed to skip writing the (redundant) merged `.npz` to disk, since STEP 4 immediately re-saves the data split by train/val/test anyway.

In [ ]:
# =============================================================================
# STEP 2 — Merge all per-file checkpoint chunks into a single in-memory
# (X, y, event_id, feature_names) tuple.
#
# output_file="" : skip writing a merged .npz to disk (STEP 4 below saves
#                   the train/val/test splits separately anyway, so an
#                   intermediate merged copy would just be redundant I/O).
# delete_chunks_after=False : keep the per-file chunks on disk, so STEP 1
#                   remains re-runnable/resumable without re-reading the
#                   .root files, in case the split/save logic below needs
#                   to be re-run from scratch.
# =============================================================================

X_all, y_all, event_id_all, feature_names = pair_builder.merge_pair_checkpoint_chunks(
    DATASETS_PATH,
    PAIR_SAVE_NAME,
    output_file="",
    delete_chunks_after=False,
    verbose=True,
)

print("\nfeature_names:", feature_names)


In [ ]:
# STEP 2b — debug: overall shapes, per-class counts (FF/FT/TF/TT), unique events

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("event_id_all shape:", event_id_all.shape)

n_unique_events = len(np.unique(event_id_all))
print(f"\nUnique events represented in the pair dataset: {n_unique_events}")

INDEX_TO_LABEL = {v: k for k, v in LABEL_INDEX_MAP.items()}
print("\nPer-class pair counts:")
for idx in sorted(INDEX_TO_LABEL):
    count = int(np.sum(y_all == idx))
    frac = count / y_all.shape[0]
    print(f"  {INDEX_TO_LABEL[idx]:>2} (y={idx}): {count:>10}  ({100.0 * frac:.3f}%)")

assert set(np.unique(y_all)) <= set(LABEL_INDEX_MAP.values()), \
    "y_all contains label indices outside LABEL_INDEX_MAP!"


## 5. STEP 3 — Train / validation / test split without event-level data leakage

`split_pairs_by_event` splits on **unique `event_id` values**, not on individual pairs: every (jet, tau) pair belonging to the same event ends up in the same split. This differs from the H5 pipeline (which assigned whole *files* to a split): here, `build_pair_dataset_from_root` already merges all `.root` files into a single checkpoint, so the split happens directly on the merged pair dataset — no separate "file assignment" step is needed (point 5 from the proposal).

In [ ]:
# =============================================================================
# STEP 3 — Split into train / val / test by EVENT (no pair from the same
# event can leak across splits).
# =============================================================================

train_idx, val_idx, test_idx = pair_builder.split_pairs_by_event(
    event_id_all,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=RANDOM_SEED,
)

print(f"Pairs -> train: {train_idx.size} | val: {val_idx.size} | test: {test_idx.size}")
print(f"Total pairs accounted for: {train_idx.size + val_idx.size + test_idx.size} / {y_all.shape[0]}")


In [ ]:
# STEP 3b — sanity checks: no event_id leakage across splits + per-split
# class distribution.

train_events = set(np.unique(event_id_all[train_idx]).tolist())
val_events = set(np.unique(event_id_all[val_idx]).tolist())
test_events = set(np.unique(event_id_all[test_idx]).tolist())

assert train_events.isdisjoint(val_events), "LEAKAGE: events shared between train and val!"
assert train_events.isdisjoint(test_events), "LEAKAGE: events shared between train and test!"
assert val_events.isdisjoint(test_events), "LEAKAGE: events shared between val and test!"
print("[OK] no event_id overlap between train / val / test")

print(f"\nUnique events -> train: {len(train_events)} | val: {len(val_events)} | test: {len(test_events)} "
      f"| total: {len(train_events) + len(val_events) + len(test_events)} / {n_unique_events}")

for split_name, idx in (("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)):
    print(f"\n--- {split_name} class distribution ---")
    y_split = y_all[idx]
    for label_idx in sorted(INDEX_TO_LABEL):
        count = int(np.sum(y_split == label_idx))
        frac = count / y_split.shape[0] if y_split.shape[0] else 0.0
        print(f"  {INDEX_TO_LABEL[label_idx]:>2}: {count:>10}  ({100.0 * frac:.3f}%)")


## 6. STEP 4 — Materialize the splits to disk

Each split is written as plain (uncompressed) `.npy` files for `X`/`y`/`event_id`, consistent with the H5 pipeline's convention (`np.save`, not `np.savez_compressed`, for the large arrays — cheap to load back with `load_mmap`/`np.load(..., mmap_mode="r")` later on). `feature_names` is saved once, since it's identical across splits.

In [ ]:
# =============================================================================
# STEP 4 — Save X / y / event_id for each split as plain .npy files, plus
# feature_names once (identical across splits).
# =============================================================================

SPLITS = {"train": train_idx, "val": val_idx, "test": test_idx}

for split_name, idx in SPLITS.items():
    x_path = os.path.join(DATASETS_PATH, f"X_{split_name}.npy")
    y_path = os.path.join(DATASETS_PATH, f"y_{split_name}.npy")
    event_id_path = os.path.join(DATASETS_PATH, f"event_id_{split_name}.npy")

    np.save(x_path, X_all[idx])
    np.save(y_path, y_all[idx])
    np.save(event_id_path, event_id_all[idx])

    print(f"[OK] {split_name}: saved X{X_all[idx].shape}, y{y_all[idx].shape}, "
          f"event_id{event_id_all[idx].shape}")

np.savez_compressed(
    os.path.join(DATASETS_PATH, "pair_feature_names.npz"),
    feature_names=np.array(feature_names, dtype=object),
    label_index_map=np.array(list(LABEL_INDEX_MAP.items()), dtype=object),
    dr_thr=DR_THR,
)
print("\n[OK] feature_names / label_index_map / dr_thr saved to pair_feature_names.npz")


In [ ]:
# Free the large in-memory arrays before standardization (STEP 5): from
# here on, each split is reloaded from disk as needed, exactly as in the
# H5 pipeline's memory-management convention.

del X_all, y_all, event_id_all
gc.collect()
